In [ ]:
import numpy as np
import json
import plotly.graph_objects as go
from IPython.display import display, HTML

In [ ]:
# Colors as RGB tuples (0-1 range)
colors = {
    "circulation": (0.2, 0.6, 0.9),      # Blue
    "open_work_area": (0.9, 0.7, 0.2),   # Yellow/Orange
    "kitchen": (0.9, 0.3, 0.3),          # Red
    "bathroom_men": (0.3, 0.8, 0.5),     # Green
    "bathroom_women": (0.8, 0.4, 0.8),   # Purple
    "reception": (0.5, 0.5, 0.9),        # Light blue
    "storage": (0.6, 0.6, 0.6),          # Gray
    "meeting": (0.3, 0.9, 0.9)           # Cyan
}

def get_color_for_space(space_type, space_id):
    """Get color for a space type, or generate one based on space_id."""
    if space_type in colors:
        return colors[space_type]
    # Generate a consistent color based on space_id hash
    h = hash(space_id)
    r = ((h >> 16) & 0xFF) / 255.0
    g = ((h >> 8) & 0xFF) / 255.0
    b = (h & 0xFF) / 255.0
    return (r, g, b)

In [ ]:
def create_3d_grid(data, grid_size=15, cell_height=0.5):
    """
    Create an interactive 3D grid visualization using plotly.
    """
    fig = go.Figure()
    
    # Process each space
    for space in data["allocation"]:
        space_type = space['type']
        space_name = space.get('name', space['space_id'])
        color = get_color_for_space(space_type, space['space_id'])
        color_str = f'rgb({int(color[0]*255)}, {int(color[1]*255)}, {int(color[2]*255)})'
        
        # Collect cell positions for this space
        for cell_id in space['cell_ids']:
            row, col = map(int, cell_id.split(';'))
            
            # Create a cube for each cell
            x, y, z = col, row, 0
            
            # Define cube vertices
            cube_x = [x, x+1, x+1, x, x, x+1, x+1, x]
            cube_y = [y, y, y+1, y+1, y, y, y+1, y+1]
            cube_z = [0, 0, 0, 0, cell_height, cell_height, cell_height, cell_height]
            
            # Add cube as mesh3d
            fig.add_trace(go.Mesh3d(
                x=cube_x,
                y=cube_y,
                z=cube_z,
                i=[0, 0, 0, 0, 4, 4, 0, 1, 1, 2, 2, 3],
                j=[1, 2, 4, 5, 5, 6, 1, 2, 5, 3, 6, 0],
                k=[2, 3, 5, 6, 6, 7, 4, 5, 6, 6, 7, 4],
                color=color_str,
                opacity=1.0,
                name=space_name,
                showlegend=False,
                hoverinfo='name'
            ))
    
    # Update layout
    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[-0.5, grid_size+0.5], title='Column'),
            yaxis=dict(range=[-0.5, grid_size+0.5], title='Row'),
            zaxis=dict(range=[0, 2], title=''),
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=0.2),
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.0)
            )
        ),
        width=800,
        height=600,
        margin=dict(l=0, r=0, t=30, b=0),
        title='Grid Layout Visualization'
    )
    
    return fig

In [ ]:
def create_legend(data):
    """Create a simple legend showing space types and their colors."""
    from IPython.display import HTML
    
    # Collect unique space types from the data
    space_info = {}
    for space in data["allocation"]:
        space_type = space['type']
        color = get_color_for_space(space_type, space['space_id'])
        if space_type not in space_info:
            space_info[space_type] = color
    
    # Generate HTML legend
    html = '<div style="display: flex; flex-wrap: wrap; gap: 10px; margin: 10px 0;">'
    for space_type, color in space_info.items():
        rgb = f'rgb({int(color[0]*255)}, {int(color[1]*255)}, {int(color[2]*255)})'
        html += f'''
        <div style="display: flex; align-items: center; gap: 5px;">
            <div style="width: 20px; height: 20px; background-color: {rgb}; border: 1px solid #333;"></div>
            <span>{space_type}</span>
        </div>'''
    html += '</div>'
    
    return HTML(html)

In [ ]:
# Load data
file_path = "/Users/benjaminherrmann/Desktop/Projects/thesis/thesis-data/var/sanitized_inference_results.jsonl"
with open(file_path, "r") as f:
    lines = f.readlines()
    print(f"Number of lines in the file: {len(lines)}")

data = json.loads(lines[0])["response"]

# Print space information
print("\nSpaces in layout:")
for space in data["allocation"]:
    space_type = space['type']
    space_name = space.get('name', space['space_id'])
    print(f"  - {space_name} ({space_type}): {len(space['cell_ids'])} cells")

In [ ]:
# Display legend
display(create_legend(data))

# Create and display 3D visualization
fig = create_3d_grid(data)

# Render as HTML for VSCode compatibility
display(HTML(fig.to_html(include_plotlyjs='cdn')))